## 🌿 Krishi AI — Plant Disease Training
### Step 1: Install Dependencies

In [ ]:
!pip install -q kagglehub tqdm torch torchvision pillow opencv-python-headless requests
print('✅ All dependencies installed!')

### Step 2: Imports

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, ConcatDataset
from tqdm.notebook import tqdm
import requests
import zipfile
print('✅ Imports done!')

### Step 3: Configuration
> ✏️ Set your Ngrok URL below

In [ ]:
# ✏️ Paste your Ngrok URL here (from: ngrok http 8000)
# Example: "https://abc123.ngrok-free.app"
BACKEND_URL = "https://regain-agile-army.ngrok-free.dev"
API_KEY     = "kalhara_secure_api_2026"

GDRIVE_SAVE_PATH = "/content/drive/MyDrive/krishi_ai_models/best_plant_model.pth"
LOCAL_MODEL_PATH = "best_plant_model.pth"

print(f'Backend URL : {BACKEND_URL}')
print(f'Model path  : {LOCAL_MODEL_PATH}')

### Step 4: Mount Google Drive (saves model permanently)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(os.path.dirname(GDRIVE_SAVE_PATH), exist_ok=True)
    print('✅ Google Drive mounted. Model will be saved to:', GDRIVE_SAVE_PATH)
    USE_GDRIVE = True
except Exception as e:
    print(f'⚠️  Google Drive not mounted: {e}')
    print('    Model will only be saved locally (lost when session ends).')
    USE_GDRIVE = False

### Step 5: Download Verified Data from Backend (Continuous Learning)

In [ ]:
live_data_dir = './live_data'
headers = {"x-api-key": API_KEY}

print(f'Connecting to backend at {BACKEND_URL} ...')
try:
    response = requests.get(f"{BACKEND_URL}/api/export-data", headers=headers, timeout=30)
    if response.status_code == 200:
        with open('verified_data.zip', 'wb') as f:
            f.write(response.content)
        with zipfile.ZipFile('verified_data.zip', 'r') as zip_ref:
            zip_ref.extractall(live_data_dir)
        print('✅ Downloaded user-verified data from backend!')
    elif response.status_code == 404:
        print('ℹ️  No verified data on backend yet — training with Kaggle data only.')
    else:
        print(f'⚠️  Backend returned {response.status_code}. Training with Kaggle data only.')
except Exception as e:
    print(f'⚠️  Could not connect to backend: {e}')
    print('    Training with Kaggle data only.')

### Step 6: Model Architecture (EfficientNet-B0)

In [ ]:
class PlantDiseaseClassifier(nn.Module):
    def __init__(self, num_classes=38):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()

        self.fc1 = nn.Linear(in_features, 512)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(512, 128)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = self.fc1(x);  x = self.relu1(x);  x = self.drop1(x)
        x = self.fc2(x);  x = self.relu2(x);  x = self.drop2(x)
        x = self.fc3(x)
        return x

print('✅ Model architecture defined!')

### Step 7: Download Dataset & Prepare DataLoaders

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Downloading Plant Disease Dataset from Kaggle...')
import kagglehub
dataset_path = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')
print(f'✅ Dataset downloaded to: {dataset_path}')

# Find train/valid folders regardless of nesting
train_path = val_path = None
for root, dirs, files in os.walk(dataset_path):
    if 'train' in dirs and 'valid' in dirs:
        train_path = os.path.join(root, 'train')
        val_path   = os.path.join(root, 'valid')
        break
assert train_path and os.path.exists(train_path), 'Train folder not found!'
assert val_path   and os.path.exists(val_path),   'Valid folder not found!'
print(f'Train: {train_path}')
print(f'Valid: {val_path}')

train_dataset = datasets.ImageFolder(root=train_path, transform=transform)
val_dataset   = datasets.ImageFolder(root=val_path,   transform=transform)
class_names   = train_dataset.classes
print(f'✅ {len(class_names)} classes, {len(train_dataset):,} training images, {len(val_dataset):,} validation images.')

# Merge with backend verified data if available
try:
    if os.path.exists(live_data_dir) and len(os.listdir(live_data_dir)) > 0:
        live_dataset = datasets.ImageFolder(live_data_dir, transform=transform)
        full_train = ConcatDataset([train_dataset, live_dataset])
        print(f'✅ Merged Kaggle + {len(live_dataset):,} verified backend images!')
    else:
        full_train = train_dataset
        print('ℹ️  No live data to merge.')
except Exception as e:
    print(f'⚠️  Live data load failed: {e}')
    full_train = train_dataset

train_loader = DataLoader(full_train,    batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print('✅ DataLoaders ready!')

print('\n📋 CLASS NAMES (copy into backend/app/config.py):')
print(class_names)

### Step 8: Train the Model
> 📊 Watch the live progress bars below — one bar per batch, summary per epoch

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Training on: {device}')
if device.type == 'cpu':
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU for faster training.')

model     = PlantDiseaseClassifier(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ✅ FIX: history dict defined BEFORE the training loop
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

epochs        = 50
patience      = 5
best_val_loss = float('inf')
counter       = 0

epoch_bar = tqdm(range(epochs), desc='Overall Progress', unit='epoch')

for epoch in epoch_bar:
    # ── Training phase ──────────────────────────────────────────────
    model.train()
    train_loss = train_correct = 0

    # ✅ tqdm progress bar per batch during training
    train_bar = tqdm(train_loader, desc=f'  Epoch {epoch+1:02d} TRAIN',
                     leave=False, unit='batch')
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * images.size(0)
        _, predicted   = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()

        # Live loss shown in the batch bar
        batch_acc = (predicted == labels).float().mean().item()
        train_bar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{batch_acc:.3f}')

    # ── Validation phase ─────────────────────────────────────────────
    model.eval()
    val_loss = val_correct = 0

    val_bar = tqdm(val_loader, desc=f'  Epoch {epoch+1:02d} VALID',
                   leave=False, unit='batch')
    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_bar.set_postfix(loss=f'{loss.item():.4f}')

    # ── Metrics ───────────────────────────────────────────────────────
    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_val_loss   = val_loss   / len(val_loader.dataset)
    train_acc      = train_correct / len(train_loader.dataset)
    val_acc        = val_correct   / len(val_loader.dataset)

    # ✅ FIX: append to history INSIDE the epoch loop
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Update outer epoch bar with summary
    epoch_bar.set_postfix(
        tr_loss=f'{avg_train_loss:.4f}', tr_acc=f'{train_acc:.3f}',
        vl_loss=f'{avg_val_loss:.4f}',  vl_acc=f'{val_acc:.3f}'
    )

    # ── Early stopping & checkpoint ───────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), LOCAL_MODEL_PATH)
        if USE_GDRIVE:
            torch.save(model.state_dict(), GDRIVE_SAVE_PATH)
            tqdm.write(f'  ✅ Epoch {epoch+1}: best model saved locally + Google Drive  (val_loss={avg_val_loss:.4f})')
        else:
            tqdm.write(f'  ✅ Epoch {epoch+1}: best model saved locally  (val_loss={avg_val_loss:.4f})')
        counter = 0
    else:
        counter += 1
        tqdm.write(f'  ⚠️  Epoch {epoch+1}: no improvement ({counter}/{patience})')
        if counter >= patience:
            tqdm.write('  🛑 Early stopping triggered.')
            break

print(f'\n🏆 Training complete! Best val loss: {best_val_loss:.4f}')

### Step 9: Training History Charts

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'],   label='Val Loss')
plt.title('Loss over Epochs'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'],   label='Val Accuracy')
plt.title('Accuracy over Epochs'); plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=120)
plt.show()
print('✅ Chart saved as training_history.png')

### Step 10: Upload Trained Model to Backend

In [ ]:
# ✅ FIX: condition was inverted — now correctly uploads when URL IS set
print('Uploading model to backend...')
try:
    with open(LOCAL_MODEL_PATH, 'rb') as f:
        files = {'file': ('latest_model.pth', f, 'application/octet-stream')}
        res = requests.post(
            f"{BACKEND_URL}/api/upload-model",
            headers=headers,
            files=files,
            timeout=180
        )
    if res.status_code == 200:
        print('✅ Model uploaded to backend successfully!')
        print('   Backend response:', res.json())
        print('   🎉 The Flutter app can now diagnose plant diseases!')
    elif res.status_code == 401:
        print('❌ Upload rejected: wrong API key.')
        print(f'   Check API_KEY matches TRAINING_API_KEY in backend/.env')
    else:
        print(f'❌ Upload failed: HTTP {res.status_code}')
        print(f'   Response: {res.text}')
        if USE_GDRIVE:
            print(f'   Model saved on Google Drive: {GDRIVE_SAVE_PATH}')
except FileNotFoundError:
    print('❌ Model file not found — did training complete successfully?')
except Exception as e:
    print(f'❌ Upload error: {e}')
    if USE_GDRIVE:
        print(f'   Model saved on Google Drive: {GDRIVE_SAVE_PATH}')

### Step 11: Quick Inference Test

In [ ]:
print('🔍 Testing model on a sample validation image...')
try:
    import glob
    from PIL import Image
    sample_images = (
        glob.glob(os.path.join(val_path, '*', '*.jpg')) +
        glob.glob(os.path.join(val_path, '*', '*.JPG')) +
        glob.glob(os.path.join(val_path, '*', '*.png'))
    )
    if sample_images:
        test_img = Image.open(sample_images[0]).convert('RGB')
        tensor   = transform(test_img).unsqueeze(0).to(device)
        model.eval()
        with torch.no_grad():
            out  = model(tensor)
            prob = torch.nn.functional.softmax(out, dim=1)
            idx  = prob.argmax().item()
        print(f'✅ Image : {os.path.basename(sample_images[0])}')
        print(f'   Predicted : {class_names[idx]}')
        print(f'   Confidence: {prob[0][idx].item()*100:.1f}%')
    else:
        print('No sample images found.')
except Exception as e:
    print(f'Test failed: {e}')